_Célula 1_

# Gráficos de usinas fotovoltaicas por tipo de região
Leitura de `UFV_consolidado.csv` e geração de gráficos por `tipo_region` (bioma, estado e país), no estilo dos gráficos de referência em `figuras/`.

In [1]:
# Célula 2
import pandas as pd
import matplotlib.pyplot as plt
import kaleido
import numpy as np
import plotly.graph_objects as go
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
import textwrap
from_drive = False
salvar_grafico = True

In [2]:
# Célula 3
if from_drive:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

In [3]:
# Célula 4
if from_drive:
    df = pd.read_csv('/content/drive/MyDrive/DL_fotovoltaica/UFV_consolidado.csv')
else:
    df = pd.read_csv('UFV_consolidado.csv')
display(df.head(6))

,year,version,area_ha,tipo_region,nome_region,sigla_region
0,2018,0-4-12-spt-4,0.617504,bioma,Amazônia,AMZ
1,2019,0-4-12-spt-4,16.561835,bioma,Amazônia,AMZ
2,2020,0-4-12-spt-4,20.086309,bioma,Amazônia,AMZ
3,2021,0-4-12-spt-4,24.763521,bioma,Amazônia,AMZ
4,2022,0-4-12-spt-4,25.998181,bioma,Amazônia,AMZ
5,2023,0-4-12-spt-4,25.909769,bioma,Amazônia,AMZ


In [4]:
# Célula 5
display(df.tail(6))

,year,version,area_ha,tipo_region,nome_region,sigla_region
168,2020,0-4-12-spt-4,7537.182891,pais,Brasil,BR
169,2021,0-4-12-spt-4,10899.019094,pais,Brasil,BR
170,2022,0-4-12-spt-4,18408.076642,pais,Brasil,BR
171,2023,0-4-12-spt-4,27829.060216,pais,Brasil,BR
172,2024,0-4-12-spt-4,37641.325534,pais,Brasil,BR
173,2025,0-4-12-spt-4,43258.458059,pais,Brasil,BR


In [5]:
# Célula 6
df[df['tipo_region'] == 'bioma'].nome_region.unique()

array(['Amazônia', 'Caatinga', 'Cerrado', 'Mata Atlântica', 'Pampa'],
      dtype=object)

In [6]:
# Célula 7
# Paleta sequencial usada em todos os gráficos (clara = valor menor/ano mais
# antigo, escura = valor maior/ano mais recente) — tons de cinza definidos
# pelo usuário (#757272, #696666, #5d5b5b, #514f4f), completados com 3 tons
# mais claros seguindo a mesma progressão.
# CMAP_CINZA = mcolors.LinearSegmentedColormap.from_list(
#     'ufv_cinza', ['#999696', '#8d8a8a', '#817e7e', '#757272', '#696666', '#5d5b5b', '#514f4f']
# )
CMAP_LARANJA_ESTADOS = mcolors.LinearSegmentedColormap.from_list(
    'eolica_laranja', ['#F7A173', '#F3732F', '#F05D0E', '#CD500C', "#B3470D", "#953A08", "#7F3006"]
)

def rampa_cores(n: int, inverso: bool = False):
    """`n` cores igualmente espaçadas na paleta sequencial (clara → escura)."""
    posicoes = np.linspace(0.05, 0.95, n)
    if inverso:
        posicoes = posicoes[::-1]
    return [CMAP_LARANJA_ESTADOS(p) for p in posicoes]


def formata_ptbr(valor, casas: int = 0) -> str:
    """Formata número no padrão brasileiro (ponto como separador de milhar)."""
    s = f'{valor:,.{casas}f}'
    return s.replace(',', '_').replace('.', ',').replace('_', '.')


def quebra_nome(nome: str, largura: int = 12) -> str:
    """Quebra nomes de região longos em 2 linhas para caber no eixo X."""
    return '\n'.join(textwrap.wrap(nome, width=largura, break_long_words=False))


FORMATADOR_PTBR = FuncFormatter(lambda v, _: formata_ptbr(v))

print('Utilitários de estilo definidos.')

Utilitários de estilo definidos.


In [7]:
# Célula 8
# Requer o pacote `kaleido` instalado (pip install -U kaleido) para exportar
# as figuras Plotly como imagem.
try:
    from google.colab import files as colab_files
    EM_COLAB = True
except ImportError:
    EM_COLAB = False


def salvar_grafico_alta_resolucao(fig, nome_arquivo, dpi=300, formato='png', baixar=True):
    """Salva um gráfico Plotly (`fig`) em alta resolução (300 ou 450 dpi) e,
    se `baixar=True` e estiver rodando no Colab, dispara o download do
    arquivo.

    O Plotly/kaleido não trabalha com DPI diretamente — a resolução da
    imagem exportada é controlada por `scale` (multiplicador do
    width/height definidos no layout da figura). Aqui `scale` é calculado
    como dpi/96, usando 96 dpi como referência padrão de tela.
    """
    escala = dpi / 96
    caminho = nome_arquivo if nome_arquivo.lower().endswith(f'.{formato}') else f"{nome_arquivo}.{formato}"
    fig.write_image(caminho, scale=escala)
    print(f"Gráfico salvo em '{caminho}' (dpi≈{dpi}, scale={escala:.2f})")

    if baixar and EM_COLAB:
        colab_files.download(caminho)


# uso: salvar_grafico_alta_resolucao(fig, 'evolucao_area_por_bioma', dpi=300)
# ou, para mais nitidez: dpi=450

_Célula 9_

## Resumo
Painel duplo — evolução da área no Brasil (com destaque do valor mais recente) ao lado da participação por bioma, lado a lado na mesma figura.

In [8]:
# Célula 10
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
totais_anuais = pivot_bioma.sum(axis=1)

cores_map = {'Caatinga': "#AB420A", 'Cerrado': "#BE4E12", 'Mata Atlântica': "#F27836", 'Amazônia': '#F7A173', 'Pampa': "#F2B99B"}

# Só Caatinga e Cerrado mostram rótulo (valor + %), em todos os anos a partir
# de 2021, para dar pra acompanhar o histórico ano a ano. Os demais biomas
# (Mata Atlântica, Amazônia, Pampa) ficam sem rótulo nas barras.
BIOMAS_HISTORICO = ['Caatinga', 'Cerrado']

fig = go.Figure()

for bioma in pivot_bioma.columns:
    textos = []
    for ano, v in zip(pivot_bioma.index, pivot_bioma[bioma]):
        if bioma in BIOMAS_HISTORICO and ano >= 2021 and v > 0:
            valor_k = f"{v/1000:.1f} K".replace('.', ',')
            pct = (v / totais_anuais.loc[ano]) * 100
            textos.append(f"{valor_k}<br>{pct:.0f}%")
        else:
            textos.append("")

    fig.add_trace(go.Bar(
        x=pivot_bioma.index, y=pivot_bioma[bioma], name=bioma,
        marker_color=cores_map.get(bioma, '#ccc'),
        text=textos, textposition='inside', insidetextanchor='middle',
        textfont=dict(color='white', size=14),
        hovertemplate='<b>Bioma:</b> ' + bioma + '<br><b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

# Rótulo do total (soma de todas as áreas) acima de cada barra, em todas as
# barras/anos — para manter o histórico completo visível, não só 2021+.
textos_totais = [f"<b>{v/1000:.1f} K</b>".replace('.', ',') for v in totais_anuais]
fig.add_trace(go.Scatter(
    x=pivot_bioma.index, y=totais_anuais, mode='text', text=textos_totais,
    textposition='top center', textfont=dict(size=15, color='black'),
    showlegend=False, hoverinfo='skip'
))

fig.update_layout(barmode='stack', title_text="Evolução da Área por Bioma", template="plotly_white", width=900, height=650,
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'),
    margin=dict(t=80, b=150, l=60, r=60), xaxis=dict(tickmode='linear'), yaxis=dict(range=[0, totais_anuais.max() * 1.2]))
fig.show()
if salvar_grafico: 
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_area_por_bioma', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_area_por_bioma.png' (dpi≈450, scale=4.69)


In [10]:
# Célula 11

In [9]:
# Célula 12
ANO_INICIAL, ANO_FINAL = 2016, 2025


def calcula_crescimento(df, tipo_region, ano_inicial=ANO_INICIAL, ano_final=ANO_FINAL):
    """Área e % de crescimento entre `ano_inicial` e `ano_final` para cada
    região do `tipo_region` informado ('bioma', 'estado' ou 'pais')."""
    sub = df[df['tipo_region'] == tipo_region]
    pivot = sub.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)

    area_inicial = pivot.loc[ano_inicial]
    area_final = pivot.loc[ano_final]

    tabela = pd.DataFrame({
        'nome_region': area_inicial.index,
        f'area_{ano_inicial}': area_inicial.values,
        f'area_{ano_final}': area_final.values,
    })
    # np.where evita divisão por zero nas regiões que ainda não tinham área em ano_inicial
    tabela['crescimento_pct'] = np.where(
        tabela[f'area_{ano_inicial}'] > 0,
        (tabela[f'area_{ano_final}'] / tabela[f'area_{ano_inicial}'] - 1) * 100,
        np.nan,
    )
    return tabela.sort_values('crescimento_pct', ascending=False, na_position='last').reset_index(drop=True)


def formata_tabela_crescimento(tabela):
    tabela_fmt = tabela.copy()
    for col in tabela_fmt.columns:
        if col.startswith('area_'):
            tabela_fmt[col] = tabela_fmt[col].apply(formata_ptbr)
    tabela_fmt['crescimento_pct'] = tabela_fmt['crescimento_pct'].apply(
        lambda p: f"{p:.0f}%".replace('.', ',') if pd.notna(p) else "—"
    )
    return tabela_fmt


# Crescimento nacional (país)
crescimento_pais = calcula_crescimento(df, 'pais')
pct_brasil = crescimento_pais['crescimento_pct'].iloc[0]
print(f"Crescimento da área de UFV no Brasil entre {ANO_INICIAL} e {ANO_FINAL}: "
      + f"{pct_brasil:.0f}%".replace('.', ','))
display(formata_tabela_crescimento(crescimento_pais))

# Crescimento por estado
print(f"\nCrescimento por estado entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'estado')))

# Crescimento por bioma
print(f"\nCrescimento por bioma entre {ANO_INICIAL} e {ANO_FINAL}:")
display(formata_tabela_crescimento(calcula_crescimento(df, 'bioma')))

Crescimento da área de UFV no Brasil entre 2016 e 2025: 3751%


,nome_region,area_2016,area_2025,crescimento_pct
0,Brasil,1.123,43.258,3751%



Crescimento por estado entre 2016 e 2025:


,nome_region,area_2016,area_2025,crescimento_pct
0,Ceará,7,4.315,59632%
1,Minas Gerais,121,16.258,13297%
2,Piauí,349,4.947,1317%
3,Bahia,646,6.785,951%
4,Rio Grande do Sul,0,0,-100%
5,Espírito Santo,0,0,—
6,Paraná,0,0,—
7,Paraíba,0,1.354,—
8,Pernambuco,0,3.099,—
9,Rio Grande do Norte,0,4.350,—



Crescimento por bioma entre 2016 e 2025:


,nome_region,area_2016,area_2025,crescimento_pct
0,Caatinga,555,27.174,4796%
1,Cerrado,568,13.891,2344%
2,Pampa,0,2,675%
3,Amazônia,0,31,—
4,Mata Atlântica,0,2.163,—


In [10]:
# Célula 13
import plotly.graph_objects as go

# 1. Preparação dos dados para o último ano disponível
df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)

# Cores consistentes
# cores_map = {
#     'Caatinga': '#b23a2f',
#     'Cerrado': '#d9702f',
#     'Mata Atlântica': '#f2a860',
#     'Amazônia': '#fbe0c4',
#     'Pampa': '#7a2618'
# }

# 2. Criar Gráfico de Pizza
# rotation=120: startangle original era 90. Adicionando 30 graus anti-horário = 120.
fig = go.Figure(data=[go.Pie(
    labels=serie_ultimo_ano.index,
    values=serie_ultimo_ano.values,
    marker=dict(colors=[cores_map.get(b) for b in serie_ultimo_ano.index]),
    textinfo='percent+label+value',
    rotation=30,
    # texttemplate: ",.0f" exibe o número inteiro com separador de milhar
    texttemplate='%{label}<br>%{percent:.0%}<br>%{value:,.2f} ha',
    insidetextfont=dict(size=20),
    outsidetextfont=dict(size=20),
    hovertemplate='<b>Bioma:</b> %{label}<br><b>Área:</b> %{value:,.2f} ha<extra></extra>'
)])

# 3. Layout
fig.update_layout(
    title_text=f"Participação por Bioma na Área de Usinas Fotovoltaicas ({ultimo_ano})",
    title_font=dict(size=24),
    template="plotly_white",
    width = 800,
    height = 700,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.35,
        xanchor="center",
        x=0.5,
        entrywidth=0.3,
        entrywidthmode='fraction',
        font=dict(size=16)
    ),
    margin=dict(t=60, b=100, l=50, r=50)
)

fig.show()
if salvar_grafico: 
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'proporsao_area_por_bioma_2025', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporsao_area_por_bioma_2025.png' (dpi≈450, scale=4.69)


In [11]:
# Célula 13A
# Anéis concêntricos (não fatias de uma mesma pizza): cada bioma ocupa uma
# faixa de raio própria, do maior para o menor (mais externo -> mais
# interno), e todos os arcos partem do mesmo ângulo inicial (0°/12h), com o
# comprimento proporcional à participação do bioma no total de UFV. Por isso
# os arcos ficam com tamanhos bem diferentes entre si (esperado).
import plotly.graph_objects as go

df_bioma = df[df['tipo_region'] == 'bioma']
pivot_bioma = df_bioma.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano = int(pivot_bioma.index.max())
serie_ultimo_ano = pivot_bioma.loc[ultimo_ano].sort_values(ascending=False)
total_ano = serie_ultimo_ano.sum()

# cores_map já definido na Célula 10 (reaproveitado aqui, como na Célula 13):
# cores_map = {'Caatinga': '#514f4f', 'Cerrado': '#696666', 'Mata Atlântica': '#817e7e', 'Amazônia': '#8d8a8a', 'Pampa': '#999696'}

# Ordem dos anéis (do mais externo ao mais interno) calculada dinamicamente a
# partir dos dados — os 5 biomas de UFV (incluindo Amazônia), não fixos em 4
# nomes como na versão da Eólica.
ORDEM_ANEIS = serie_ultimo_ano.index.tolist()
N = len(ORDEM_ANEIS)
ESPACO = 0.02  # respiro entre um anel e o próximo
ESPESSURA = (1 - (N - 1) * ESPACO) / N  # espessura de cada anel, para o conjunto ocupar de r=0 até r=1

fig_aneis = go.Figure()
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    arco_graus = frac * 360
    base_raio = 1 - i * (ESPESSURA + ESPACO) - ESPESSURA

    fig_aneis.add_trace(go.Barpolar(
        r=[ESPESSURA],
        theta=[arco_graus / 2],  # todos os arcos começam em 0°; o centro do arco é a metade do seu próprio comprimento
        width=[arco_graus],
        base=[base_raio],
        name=bioma,
        marker=dict(color=cores_map.get(bioma, '#ccc'), line=dict(color='white', width=1)),
        hovertemplate=f'<b>Bioma:</b> {bioma}<br><b>Área:</b> {valor:,.2f} ha<br><b>Participação:</b> {frac*100:.1f}%<extra></extra>',
    ))

# --- rótulos: lista empilhada à esquerda do anel, texto grande e em negrito,
# tamanho decrescente conforme a posição no ranking (não fixo por nome de
# bioma, já que UFV tem 5 biomas em vez dos 4 da Eólica) ---
TAMANHOS_FONTE = np.linspace(30, 15, N)

anotacoes = []
Y_INICIAL, PASSO_Y = 0.90, 0.72 / max(N - 1, 1)
for i, bioma in enumerate(ORDEM_ANEIS):
    valor = serie_ultimo_ano.get(bioma, 0)
    frac = valor / total_ano
    anotacoes.append(dict(
        x=0.04, y=Y_INICIAL - i * PASSO_Y, xref='paper', yref='paper',
        xanchor='left', align='left',
        text=f"<b>{bioma} {frac*100:.0f}%</b>",
        showarrow=False,
        font=dict(size=int(TAMANHOS_FONTE[i]), color='#1b1a1a'),
    ))

fig_aneis.update_layout(
    title_text=f"Participação por Bioma na Área de Usinas Fotovoltaicas ({ultimo_ano})",
    title_font=dict(size=24),
    template="plotly_white",
    width=900, height=800,
    showlegend=False,
    annotations=anotacoes,
    polar=dict(
        # anel deslocado para a direita, deixando espaço à esquerda para os rótulos
        domain=dict(x=[0.38, 0.98], y=[0.05, 0.95]),
        radialaxis=dict(visible=False, range=[0, 1]),
        angularaxis=dict(visible=False, rotation=90, direction='clockwise'),
    ),
    margin=dict(t=90, b=30, l=30, r=30),
)

fig_aneis.show()
if salvar_grafico:
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig_aneis, 'proporcao_area_por_bioma_UFV_aneis_concentricos', dpi=450)

Salvando o grafico com DPI 450
Gráfico salvo em 'proporcao_area_por_bioma_UFV_aneis_concentricos.png' (dpi≈450, scale=4.69)


_Célula 14_

### Participação por bioma (ano mais recente)
Percentual da área de usinas fotovoltaicas por bioma no último ano disponível.

_Célula 15_

## Estados
Gráfico de barras verticais — um grupo por estado (ordenado do maior para o menor pela área do ano mais recente) e uma barra por ano dentro do grupo, com rótulo no ano mais recente — estilo do gráfico de referência em `figuras/grafico_historico_usinas.png`.

In [14]:
# Célula 16
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_estado = df[df['tipo_region'] == 'estado']
pivot_estado = df_estado.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_estado = pivot_estado.index.max()
ordem_desc = pivot_estado.loc[ultimo_ano_estado].sort_values(ascending=False).index
pivot_estado = pivot_estado[ordem_desc]

estados = pivot_estado.columns.tolist()
anos = pivot_estado.index.tolist()

idx_corte = estados.index('Paraíba') + 1
grupo1 = [e for e in estados[:idx_corte] if e != 'Tocantins']

cores_anos_plotly = [f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})' for r, g, b, a in rampa_cores(len(anos))]

# Maior valor do grupo, para dar espaço suficiente acima da barra mais alta
# (senão o Plotly encolhe automaticamente o rótulo dela para caber)
max_valor_grupo1 = max(pivot_estado.loc[ano, e] for ano in anos for e in grupo1)

fig = go.Figure()

for i, ano in enumerate(anos):
    # Regra: Destaque apenas em 2021 e 2025 com fonte 26
    font_size = 18 if ano in [2025] else 1  # 2021,
    exibir_texto = ano in [2025]   # 2021,

    valores = [pivot_estado.loc[ano, e] for e in grupo1]
    textos = [formata_ptbr(v) if (v > 0 and exibir_texto) else "" for v in valores]

    fig.add_trace(
        go.Bar(
            x=grupo1,
            y=valores,
            name=str(ano),
            marker_color=cores_anos_plotly[i],
            # text=textos,
            # textposition='outside',
            # textfont=dict(size=font_size, color='black'),
            # textangle=-90,
            # Sem isso, o Plotly encolhe o texto das barras mais altas (perto
            # do topo do gráfico) para caber no espaço disponível, deixando o
            # rótulo minúsculo/ilegível mesmo com font_size fixo em 26.
            constraintext='none',
            hovertemplate='<b>Estado:</b> %{x}<br><b>Ano:</b> ' + str(ano) + '<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
        )
    )

fig.update_layout(
    title="Estados Principais - Evolução da Área (Destaque 2021/2025)",
    width=1050, height=750, template="plotly_white", barmode='group',
    legend=dict(
        orientation="h", y=-0.05, x=0.5, xanchor="center",
        font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'
    ),
    margin=dict(t=80, b=150, l=60, r=40),
    # Espaço extra acima da barra mais alta para o rótulo vertical não ser cortado
    yaxis=dict(range=[0, max_valor_grupo1 * 1.35])
)
fig.update_xaxes(tickangle=0)
fig.show()

if salvar_grafico: 
    print("Salvando o grafico com DPI 450")
    salvar_grafico_alta_resolucao(fig, 'evolucao_anoXarea_por_estado', dpi=600)

Salvando o grafico com DPI 450
Gráfico salvo em 'evolucao_anoXarea_por_estado.png' (dpi≈600, scale=6.25)


_Célula 17_

### Participação percentual por estado (ano mais recente)
Percentual da área de usinas fotovoltaicas de cada estado em relação ao total somado de todos os estados.

In [16]:
# Célula 18
df_estado_pct = df[df['tipo_region'] == 'estado']
pivot_estado_pct = df_estado_pct.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_pct = pivot_estado_pct.index.max()

serie_estado_pct = pivot_estado_pct.loc[ultimo_ano_pct].sort_values(ascending=False)
total_estados_pct = serie_estado_pct.sum()

tabela_pct_estado = pd.DataFrame({
    'estado': serie_estado_pct.index,
    'area_ha': serie_estado_pct.values,
})
tabela_pct_estado['percentual'] = tabela_pct_estado['area_ha'] / total_estados_pct * 100

# versão formatada (pt-BR) só para exibição — tabela_pct_estado continua numérica
tabela_pct_estado_fmt = tabela_pct_estado.copy()
tabela_pct_estado_fmt['area_ha'] = tabela_pct_estado_fmt['area_ha'].apply(formata_ptbr)
tabela_pct_estado_fmt['percentual'] = tabela_pct_estado_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_pct_estado_fmt)

,estado,area_ha,percentual
0,Minas Gerais,16.258,"37,6%"
1,Bahia,6.785,"15,7%"
2,Piauí,4.947,"11,4%"
3,Rio Grande do Norte,4.350,"10,1%"
4,Ceará,4.315,"10,0%"
5,Pernambuco,3.099,"7,2%"
6,São Paulo,2.077,"4,8%"
7,Paraíba,1.354,"3,1%"
8,Tocantins,43,"0,1%"
9,Rondônia,31,"0,1%"


In [14]:
# Célula 19
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def adicionar_traces_v5(fig, lista_estados, row, show_leg):
    for i, ano in enumerate(anos):
        # Aplicando destaque consistente de fonte 16px para 2021/2025
        font_size = 16 if ano in [2021, 2025] else 1
        exibir_texto = ano in [2021, 2025]
        valores = [pivot_estado.loc[ano, e] for e in lista_estados]
        textos = [formata_ptbr(v) if (v > 0 and exibir_texto) else "" for v in valores]

        fig.add_trace(go.Bar(
            x=lista_estados, y=valores, name=str(ano),
            marker_color=cores_anos_plotly[i], text=textos, textposition='outside',
            textfont=dict(size=font_size, color='black'),
            textangle=-90, legendgroup=str(ano), showlegend=show_leg,
            hovertemplate='<b>Estado:</b> %{x}<br><b>Ano:</b> ' + str(ano) + '<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
        ), row=row, col=1)

fig = make_subplots(rows=2, cols=1, subplot_titles=('Estados Principais', 'Demais Estados'), vertical_spacing=0.2)
adicionar_traces_v5(fig, grupo1, 1, True)
grupo2 = [e for e in estados[idx_corte:] if e != 'Tocantins']
adicionar_traces_v5(fig, grupo2, 2, False)

fig.update_layout(width=950, height=1300, template="plotly_white", barmode='group',
    legend=dict(orientation="h", y=-0.05, x=0.5, xanchor="center", font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'),
    margin=dict(t=80, b=100, l=60, r=40))
fig.update_xaxes(tickangle=0)
fig.show()

In [17]:
# Célula 20
ESTADOS_GRUPO = ['Minas Gerais', 'Bahia', 'Piauí', 'Rio Grande do Norte']

df_estado_grupo = df[df['tipo_region'] == 'estado']
pivot_estado_grupo = df_estado_grupo.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum').fillna(0)
ultimo_ano_grupo = pivot_estado_grupo.index.max()

serie_estado_grupo = pivot_estado_grupo.loc[ultimo_ano_grupo]
total_geral_estados = serie_estado_grupo.sum()
area_grupo = serie_estado_grupo[ESTADOS_GRUPO].sum()
percentual_grupo = area_grupo / total_geral_estados * 100

tabela_grupo_estados = pd.DataFrame({
    'estados': [' + '.join(ESTADOS_GRUPO)],
    'area_ha': [area_grupo],
    'percentual': [percentual_grupo],
})

# versão formatada (pt-BR) só para exibição
tabela_grupo_estados_fmt = tabela_grupo_estados.copy()
tabela_grupo_estados_fmt['area_ha'] = tabela_grupo_estados_fmt['area_ha'].apply(formata_ptbr)
tabela_grupo_estados_fmt['percentual'] = tabela_grupo_estados_fmt['percentual'].apply(lambda p: f"{p:.1f}%".replace('.', ','))

display(tabela_grupo_estados_fmt)

,estados,area_ha,percentual
0,Minas Gerais + Bahia + Piauí + Rio Grande do N...,32.340,"74,8%"


In [ ]:
# Célula 21

_Célula 22_

## País (Brasil)
Como há apenas uma região do tipo `pais` (Brasil), o gráfico mostra a área por ano — estilo do gráfico de referência em `figuras/historico_UFV_Br.png`.

In [ ]:
# Célula 23
import plotly.graph_objects as go
import numpy as np

# 1. Preparação dos dados
df_pais = df[df['tipo_region'] == 'pais']
pivot_pais = df_pais.pivot_table(index='year', columns='nome_region', values='area_ha', aggfunc='sum')
serie_brasil = pivot_pais['Brasil']

anos = serie_brasil.index.tolist()
valores = serie_brasil.values.tolist()

# Cores baseadas na rampa definida anteriormente (claro -> escuro)
cores_br = [f'rgba({int(r*255)}, {int(g*255)}, {int(b*255)}, {a})' for r, g, b, a in rampa_cores(len(anos))]

# 2. Criar Gráfico de Linha Interativo
fig = go.Figure()

# Adicionando a linha
fig.add_trace(go.Scatter(
    x=anos,
    y=valores,
    mode='lines+markers+text',
    line=dict(color='#d9702f', width=4), # Cor intermediária da paleta para a linha
    marker=dict(
        color=cores_br,
        size=12,
        line=dict(color='white', width=1.5)
    ),
    text=[formata_ptbr(v) for v in valores],
    textposition="top center",
    textfont=dict(size=11, color='#555'),
    hovertemplate='<b>Ano:</b> %{x}<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
))

# 3. Layout
fig.update_layout(
    title="Área de Usinas Fotovoltaicas no Brasil por Ano",
    xaxis_title="Ano",
    yaxis_title="Área (ha)",
    template="plotly_white",
    width=800,
    height=550,
    xaxis=dict(
        tickmode='linear',
        range=[min(anos) - 0.5, max(anos) + 0.5]
    ),
    yaxis=dict(
        range=[0, max(valores) * 1.2], # Espaço para os rótulos superiores
        tickformat=".2s",
        hoverformat=",.2f"
    ),
    margin=dict(t=80, b=60, l=60, r=40)
)

fig.show()

In [ ]:
# Célula 24
fig_to = go.Figure()
for i, ano in enumerate(anos):
    valor = pivot_estado.loc[ano, 'Tocantins']
    # Aplicando fonte 16 para 2021 e 2025
    font_size = 16 if ano in [2021, 2025] else 12
    exibir_texto = ano in [2021, 2025]

    fig_to.add_trace(go.Bar(
        x=['Tocantins'], y=[valor], name=str(ano),
        marker_color=cores_anos_plotly[i],
        text=[formata_ptbr(valor) if (valor > 0 and exibir_texto) else ""],
        textposition='outside',
        textfont=dict(size=font_size, color='black'),
        textangle=-90, legendgroup=str(ano), showlegend=True,
        hovertemplate='<b>Ano:</b> ' + str(ano) + '<br><b>Área:</b> %{y:,.2f} ha<extra></extra>'
    ))

fig_to.update_layout(title="Tocantins - Evolução", width=600, height=600, template="plotly_white", barmode='group',
    legend=dict(orientation="h", y=-0.3, x=0.5, xanchor='center', font=dict(size=16), entrywidth=0.19, entrywidthmode='fraction'),
    margin=dict(t=80, b=150, l=60, r=40))
fig_to.show()

In [ ]:
# Célula 25